In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))


### 頑張っていこう！

### Validation on APS datesets

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path

def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def fig_compare_all(folder_path):

    folder = Path(folder_path)

    # 获取文件夹内所有 json 文件，并按文件名排序

    json_files = sorted(folder.glob("*.json"))

    for json_path in json_files:

        print(f"Processing: {json_path}")

        compare(str(json_path))

def compare(path):
	fig, ax = plt.subplots(figsize=(2.2, 1.5))
#     fix ax
	ax.set_position([0.26, 0.26, 0.65, 0.65])
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		results_aps = np.array(data['results_aps'])
		stop, num = data['p_interval']
		runstep = data['runstep']
		type_ana = data['type_ana']
		year = data['year']
	ps = generate_interval((stop, num))
	results = results.reshape(num+1, runstep)
	means = results.mean(axis=1)
	results_aps = results_aps.reshape(num+1, runstep)
	means_aps = results_aps.mean(axis=1)

	plt.plot(ps, means_aps, color='#0070C0', lw=3, marker='o', 
             markersize=5, markerfacecolor='white', markeredgecolor='#0070C0', 
             markeredgewidth=1, markevery=12)
    
	plt.plot(ps, means, color='#7FBF7B', lw=3, marker='s', 
             markersize=5, markerfacecolor='white', markeredgecolor='#7FBF7B', 
             markeredgewidth=1, markevery=12)
    
	if type_ana == "ana_11":
		ax.text(
    1.0, 0.02,
    f"{year}-{year+1}",
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=11)
        
	if type_ana == "ana_12" or type_ana == "ana_2":
		ax.text(
    0.58, 0.8,
    f"{year}-{year+1}",
    transform=ax.transAxes,
    ha='right',
    va='bottom',
    fontsize=11)     


	for label in plt.gca().get_xticklabels() + plt.gca().get_yticklabels():
# 		label.set_fontweight('bold')
		label.set_fontsize(11)

    
# 	legend_elements = [
# 	Line2D([0], [0], color='#666666', lw=3, label=r'$\mathcal{H}^{\text{aps}}$',
# 		marker='o', markersize=5, markerfacecolor='white', markeredgecolor='#666666', markeredgewidth=1),
# 	Line2D([0], [0], color='#AF8DC3', lw=3, label=r'$\mathcal{H}^{\ast}$',
# 		marker='s', markersize=5, markerfacecolor='white', markeredgecolor='#AF8DC3', markeredgewidth=1)
# 	]
# 	plt.legend(handles=legend_elements, loc='upper left', fontsize=11, 
#                frameon=False, ncols=1, bbox_to_anchor=(-0.05, 1.1))

    
	filename = f"{year}" + 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
	save_dir_figs=f"figs_compareAPS/{type_ana}"
	os.makedirs(save_dir_figs, exist_ok=True)
	output_path = os.path.join(save_dir_figs, filename)
	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
# compare('results_compare_aps/ana_2/2001_ana_2_100_100t48.535330.json')

In [ ]:
fig_compare_all('results_compare_aps/ana_2/')

### Compare with ER hypergraph generation method

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path

def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def compare_er(path):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		results_er = np.array(data['results_er'])
		stop, num = data['p_interval']
		runstep = data['runstep']
		type_ana = data['type_ana']

	ps = generate_interval((stop, num))
	results = results.reshape(num+1, runstep)
	means = results.mean(axis=1)
	results_er = results_er.reshape(num+1, runstep)
	means_er = results_er.mean(axis=1)
	plt.plot(ps, means, color='#AF8DC3', lw=4, marker='s', 
             markersize=8, markerfacecolor='white', markeredgecolor='#AF8DC3', 
             markeredgewidth=2, markevery=10)
    
	plt.plot(ps, means_er, color='#666666', lw=4, marker='o', 
             markersize=8, markerfacecolor='white', markeredgecolor='#666666', 
             markeredgewidth=2, markevery=10)
        


	for label in plt.gca().get_xticklabels() + plt.gca().get_yticklabels():
		label.set_fontsize(18)


    
# 	filename = 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
# 	save_dir_figs=f"figs_compare_er/{type_ana}"
# 	os.makedirs(save_dir_figs, exist_ok=True)
# 	output_path = os.path.join(save_dir_figs, filename)
# 	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
compare_er('results_compare_er/ana_11/200_1000t17_333737.json')

### Compare with ER and BA hypergraph generation method

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path

def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def compare_er_ba(path, path_ba):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		results_er = np.array(data['results_er'])
		stop, num = data['p_interval']
		runstep = data['runstep']
		type_ana = data['type_ana']

	ps = generate_interval((stop, num))
	results = results.reshape(num+1, runstep)
	means = results.mean(axis=1)
	results_er = results_er.reshape(num+1, runstep)
	means_er = results_er.mean(axis=1)
    
	with open(path_ba, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results_ba = np.array(data['results_ba'])
		stop, num = data['p_interval']
		runstep = data['runstep']
	ps = generate_interval((stop, num))
	results_ba = results_ba.reshape(num+1, runstep)
	means_ba = results_ba.mean(axis=1)
    

    
	plt.plot(ps, means_er, color='#AF8DC3', lw=4, marker='*', 
             markersize=10, markerfacecolor='white', markeredgecolor='#AF8DC3', 
             markeredgewidth=1.5, markevery=10)
    
	plt.plot(ps, means_ba, color='#666666', lw=4, marker='v', 
             markersize=8, markerfacecolor='white', markeredgecolor='#666666', 
             markeredgewidth=1.5, markevery=15)
    
	plt.plot(ps, means, color='#7FBF7B', lw=4, marker='s', 
             markersize=8, markerfacecolor='white', markeredgecolor='#7FBF7B', 
             markeredgewidth=1.5, markevery=13)

# 	if type_ana == 'ana_12':
# 		ax.set_ylim(0.5, 14.5)
# 		ax.set_yticks([1, 7, 14])
        
# smalle inset version 
	if type_ana == 'ana_12':
		ax.set_ylim(0.95, 2.05)
		ax.set_yticks([1.0, 1.5, 2.0])
		ax.set_xticks([0.0, 0.2, 0.4])
        
	if type_ana == 'ana_11':
		ax.set_ylim(-3, 83)
		ax.set_yticks([0, 40, 80])
        
	if type_ana == 'ana_2':
		ax.set_ylim(-6, 206)
		ax.set_yticks([0, 100, 200])
        
	ax.tick_params(axis='both', labelsize=18)
# smalle inset version 
# 	ax.tick_params(axis='both', labelsize=23)
    
	filename = f"{type_ana}" 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
	save_dir_figs=f"figs_compare_erba"
	os.makedirs(save_dir_figs, exist_ok=True)
	output_path = os.path.join(save_dir_figs, filename)
	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
# 'N': 100,'T': 100,'B': 4,'R': 3
# ana_11
# compare_er_ba('results_compare_er/ana_11/200_1000t17_333737.json',
#           'results_compare_ba/ana_11/200_1000t23_598234.json')
# # ana_12
# compare_er_ba('results_compare_er/ana_12/200_1000t25_718391.json',
#           'results_compare_ba/ana_12/200_1000t03_564217.json')
# # ana_2
compare_er_ba('results_compare_er/ana_2/200_500t17_376323.json',
          'results_compare_ba/ana_2/200_1000t15_849751.json')



### Validation on ER and BA with other B values

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path

def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def compare_er_ba(path):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		results_er = np.array(data['results_er'])
		results_ba = np.array(data['results_ba'])
		stop, num = data['p_interval']
		runstep = data['runstep']
		type_ana = data['type_ana']

	ps = generate_interval((stop, num))
	results = results.reshape(num+1, runstep)
	means = results.mean(axis=1)
	results_er = results_er.reshape(num+1, runstep)
	means_er = results_er.mean(axis=1)
	results_ba = results_ba.reshape(num+1, runstep)
	means_ba = results_ba.mean(axis=1)
    
    
	plt.plot(ps, means_er, color='#AF8DC3', lw=4, marker='*', 
             markersize=10, markerfacecolor='white', markeredgecolor='#AF8DC3', 
             markeredgewidth=1.5, markevery=10)
    
	plt.plot(ps, means_ba, color='#666666', lw=4, marker='v', 
             markersize=8, markerfacecolor='white', markeredgecolor='#666666', 
             markeredgewidth=1.5, markevery=15)
    

    
	plt.plot(ps, means, color='#7FBF7B', lw=4, marker='s', 
             markersize=8, markerfacecolor='white', markeredgecolor='#7FBF7B', 
             markeredgewidth=1.5, markevery=13)
    
# 	if type_ana == 'ana_12':
# 		ax.set_ylim(0.5, 14.5)
# 		ax.set_yticks([1, 7, 14])

	if type_ana == 'ana_11':
		ax.set_ylim(-3, 83)
		ax.set_yticks([0, 40, 80])
	ax.tick_params(axis='both', labelsize=18)
    
	if type_ana == 'ana_12':
		ax.set_ylim(0.95, 2.05)
		ax.set_yticks([1.0, 1.5, 2.0])
		ax.set_xticks([0.0, 0.2, 0.4])
        
        
	if type_ana == 'ana_2':
		ax.set_ylim(-6, 206)
		ax.set_yticks([0, 100, 200])
# 	ax.tick_params(axis='both', labelsize=23)


    
	filename = f"{type_ana}" 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
	save_dir_figs=f"figs_compare_erba"
	os.makedirs(save_dir_figs, exist_ok=True)
	output_path = os.path.join(save_dir_figs, filename)
	plt.savefig(output_path, dpi=800)

	plt.show()
    
def get_three_plot(paths):
    for path in paths:
        compare_er_ba(path)
        
        
         

In [ ]:
# 'N': 100,'T': 100,'B': 5,'R': 3

# compare_er_ba('results_compare_erba/ana_11/200_1000t29_158465.json')
# compare_er_ba('results_compare_erba/ana_12/200_1000t11_980433.json')
compare_er_ba('results_compare_erba/ana_2/200_1000t18_019455.json')

# # 'N': 100,'T': 100,'B': 6,'R': 3

# compare_er_ba('results_compare_erba/ana_11/200_1000t06_851028.json')
# compare_er_ba('results_compare_erba/ana_12/200_1000t16_495703.json')
compare_er_ba('results_compare_erba/ana_2/200_1000t01_311066.json')


### Compare with other search methods

In [ ]:
import random
import math
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import warnings
warnings.simplefilter('ignore')
import json
import os
from datetime import datetime
from joblib import Parallel, delayed
import hypernetx.algorithms.hypergraph_modularity as hmod
from matplotlib.lines import Line2D
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path

def generate_interval(et):
	stop, num = et
	return list(np.arange(0, stop + stop / num, stop / num))

def retrive_mean(results, num, runstep):
	results = results.reshape(num+1, runstep)
	return results.mean(axis=1)    

def retrive_H_star(path):
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
    

    

def compare_sa(path, path_mu2, path_gr):
	fig, ax = plt.subplots(figsize=(4, 2.5))
#     fix ax
	ax.set_position([0.18, 0.15, 0.75, 0.75])
    
	with open(path, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results = np.array(data['results'])
		results_sa = np.array(data['results_sa'])
		stop, num = data['p_interval']
		runstep = data['runstep']
		type_ana = data['type_ana']
        
	with open(path_mu2, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results_mu2 = np.array(data['results_sa'])    
        
	with open(path_gr, 'r', encoding='utf-8') as f:
		data = json.load(f)
		results_gr = np.array(data['results_gr'])  
        
	ps = generate_interval((stop, num))
    
	means = retrive_mean(results, num, runstep)
    
	means_sa = retrive_mean(results_sa, num, runstep)

	means_sa_mu2 = retrive_mean(results_mu2, num, runstep)
    
	means_gr = retrive_mean(results_gr, num, runstep)
    
	marker_size = 8
	marker_width = 1    
	marker_density = 10    
    
	plt.plot(ps, means_gr, color='#333333', lw=2, marker='P', 
             markersize=marker_size, markerfacecolor='white', markeredgecolor='#333333', 
             markeredgewidth=marker_width, markevery=marker_density)
    
	plt.plot(ps, means_sa, color='#284C84', lw=4, marker='o', 
             markersize=marker_size, markerfacecolor='white', markeredgecolor='#284C84', 
             markeredgewidth=marker_width, markevery=marker_density)
    
	plt.plot(ps, means_sa_mu2, color='#B8006A', lw=4, marker='^', 
             markersize=marker_size, markerfacecolor='white', markeredgecolor='#B8006A', 
             markeredgewidth=marker_width, markevery=marker_density)
    
	plt.plot(ps, means, color='#7FBF7B', lw=4, marker='s', 
             markersize=marker_size, markerfacecolor='white', markeredgecolor='#7FBF7B', 
             markeredgewidth=marker_width, markevery=marker_density)
    
# 	if type_ana == 'ana_12':
# 		ax.set_ylim(0.955, 2.05)
# 		ax.set_yticks([1.0, 1.5, 2.0])
        
# 	if type_ana == 'ana_11':
# 		ax.set_ylim(-3, 67)
# 		ax.set_yticks([0, 32, 64])
        
# 	if type_ana == 'ana_2':
# 		ax.set_ylim(-6, 206)
# 		ax.set_yticks([0, 100, 200])
        

	ax.tick_params(axis='both', labelsize=18)


    
# 	filename = 't' + datetime.now().isoformat().split(":")[-1].replace(".", "_") + '.jpg'
# 	save_dir_figs=f"figs_compare_sa/{type_ana}"
# 	os.makedirs(save_dir_figs, exist_ok=True)
# 	output_path = os.path.join(save_dir_figs, filename)
# 	plt.savefig(output_path, dpi=800)

	plt.show()
         

In [ ]:
# # 100 100 B=4 3
# # ana_11
# compare_sa('results_compare_sa/ana_11/200_1000t26_919477.json',
#           'results_single_mu2/ana_11/200_1000t03_052832.json',
#           'results_single_greedy/ana_11/200_1000t02_980236.json')

# # ana_12
compare_sa('results_compare_sa/ana_12/200_1000t05_530290.json',
          'results_single_mu2/ana_12/200_1000t09_144374.json',
          'results_single_greedy/ana_12/200_1000t50_579754.json')

# # # ana_2
# compare_sa('results_compare_sa/ana_2/200_1000t01_483425.json',
#           'results_single_mu2/ana_2/200_1000t36_060675.json',
#           'results_single_greedy/ana_2/200_1000t41_443874.json')


# 100 100 B=5 3
# ana_11
# compare_sa('results_compare_sa/ana_11/200_1000t51_480310.json',
#           'results_single_mu2/ana_11/200_1000t53_301330.json',
#           'results_single_greedy/ana_11/200_1000t56_091920.json')

# # ana_12
compare_sa('results_compare_sa/ana_12/200_1000t48_450696.json',
          'results_single_mu2/ana_12/200_1000t38_635330.json',
          'results_single_greedy/ana_12/200_1000t06_635598.json')

# # # ana_2
# compare_sa('results_compare_sa/ana_2/200_1000t44_211002.json',
#           'results_single_mu2/ana_2/200_1000t47_899899.json',
#           'results_single_greedy/ana_2/200_1000t34_000495.json')


# # 100 100 B=6 3
# # # ana_11
# compare_sa('results_compare_sa/ana_11/200_1000t43_768868.json',
#           'results_single_mu2/ana_11/200_1000t58_247581.json',
#           'results_single_greedy/ana_11/200_1000t05_676585.json')

# # # ana_12
compare_sa('results_compare_sa/ana_12/200_1000t55_932408.json',
          'results_single_mu2/ana_12/200_1000t54_613988.json',
          'results_single_greedy/ana_12/200_1000t56_113149.json')

# # # ana_2
# compare_sa('results_compare_sa/ana_2/200_1000t56_296208.json',
#           'results_single_mu2/ana_2/200_1000t25_973930.json',
#           'results_single_greedy/ana_2/200_1000t42_818615.json')


# 500 500 B=6 3
# ana_11
# compare_sa('results_compare_sa/ana_11/200_1000t02_743989.json',
#           'results_single_mu2/ana_11/200_1000t10_875707.json',
#           'results_single_greedy/ana_11/200_1000t41_863734.json')
# # ana_12
compare_sa('results_compare_sa/ana_12/200_1000t58_261484.json',
          'results_single_mu2/ana_12/200_1000t47_112548.json',
          'results_single_greedy/ana_12/200_1000t45_364687.json')
# # # ana_2
# see f_figures_2 because I use a different method


# # # 50 50 B=6 3
# # # ana_11
# compare_sa('results_compare_sa/ana_11/200_1000t23_529317.json',
#           'results_single_mu2/ana_11/200_1000t47_025697.json',
#           'results_single_greedy/ana_11/200_1000t51_865131.json')
# # ana_12
# compare_sa('results_compare_sa/ana_12/200_1000t45_392777.json',
#           'results_single_mu2/ana_12/200_1000t22_643272.json',
#           'results_single_greedy/ana_12/200_1000t23_104381.json')
# # # ana_2
# compare_sa('results_compare_sa/ana_2/200_1000t45_750387.json',
#           'results_single_mu2/ana_2/200_1000t20_593325.json',
#           'results_single_greedy/ana_2/200_1000t39_128094.json')

# # 50 50 B=5 3
# # ana_11
# compare_sa('results_compare_sa/ana_11/.json',
#           'results_single_mu2/ana_11/.json',
#           'results_single_greedy/ana_11/.json')
# # ana_12
# compare_sa('results_compare_sa/ana_12/.json',
#           'results_single_mu2/ana_12/.json',
#           'results_single_greedy/ana_12/.json')
# # # ana_2
# compare_sa('results_compare_sa/ana_2/.json',
#           'results_single_mu2/ana_2/.json',
#           'results_single_greedy/ana_2/.json')